<a href="https://colab.research.google.com/github/eduardobatistadefreitas-art/nodo-regulator/blob/main/Projeto%20GER%20S26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
%%writefile ger_engine.py

# cole aqui o código completo do motor
"""
===============================================================================
S26_B2_core.py
Núcleo Computacional da Série S26-B

Parte 1A
Infraestrutura básica
===============================================================================
"""

import numpy as np
import scipy.linalg as la


# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

ENERGY_TOL = 1e-4
DIVERGENCE_AMPLITUDE = 1e6
EPS = 1e-15


# =============================================================================
# CONSTRUÇÃO DA REDE
# =============================================================================

def build_ring_graph(n):
    """
    Constrói o grafo periódico F1.

    Retorna
    -------
    A : matriz de adjacência
    L : Laplaciano
    theta : coordenadas angulares
    """

    A = np.zeros((n, n))

    for i in range(n):
        A[i, (i + 1) % n] = 1.0
        A[i, (i - 1) % n] = 1.0

    D = np.diag(np.sum(A, axis=1))

    L = D - A

    theta = np.linspace(
        0.0,
        2.0*np.pi,
        n,
        endpoint=False
    )

    return A, L, theta


# =============================================================================
# BASE ESPECTRAL
# =============================================================================

def spectral_basis(L):
    """
    Diagonalização do Laplaciano.
    """

    eigenvalues, eigenvectors = la.eigh(L)

    eigenvalues[np.abs(eigenvalues) < 1e-12] = 0.0

    return eigenvalues, eigenvectors


# =============================================================================
# CONDIÇÃO INICIAL
# =============================================================================

def gaussian_packet(theta,
                    center=np.pi,
                    sigma=0.10):
    """
    Pulso gaussiano inicial.
    """

    return np.exp(
        -(theta-center)**2 /
        (2*sigma**2)
    )


# =============================================================================
# POTENCIAIS
# =============================================================================

class Potential:

    @staticmethod
    def evaluate(gamma, model):

        if model == "A":

            F = gamma**2

            V = gamma**3 / 3.0

        elif model == "B":

            F = gamma**2 / (1.0 + gamma**2)

            V = gamma - np.arctan(gamma)

        elif model == "C":

            F = gamma**3

            V = gamma**4 / 4.0

        else:

            raise ValueError(
                f"Potencial desconhecido: {model}"
            )

        return F, V


# =============================================================================
# INICIALIZAÇÃO DE VERLET (O(dt²))
# =============================================================================

def initialize_verlet(
    gamma0,
    L,
    beta,
    potential,
    dt
):
    """
    Inicialização consistente de segunda ordem.
    """

    F0, _ = Potential.evaluate(
        gamma0,
        potential
    )

    accel0 = -(L @ gamma0) + beta*F0

    gamma_minus = (
        gamma0
        + 0.5*dt**2*accel0
    )

    return gamma_minus


# =============================================================================
# DETECTOR DE DIVERGÊNCIA
# =============================================================================

def check_divergence(
    gamma,
    energy_error=None
):

    amp = np.max(np.abs(gamma))

    if np.isnan(amp):
        return True

    if np.isinf(amp):
        return True

    if amp > DIVERGENCE_AMPLITUDE:
        return True

    if energy_error is not None:
        if np.isnan(energy_error):
            return True

        if np.isinf(energy_error):
            return True

    return False
# =============================================================================
# HAMILTONIANA
# =============================================================================

def compute_hamiltonian(
    gamma,
    velocity,
    L,
    beta,
    potential
):
    """
    Calcula a Hamiltoniana total do sistema.
    """

    _, V = Potential.evaluate(
        gamma,
        potential
    )

    kinetic = 0.5 * np.sum(velocity**2)

    potential_linear = (
        0.5 *
        np.dot(
            gamma,
            L @ gamma
        )
    )

    potential_nonlinear = (
        -beta *
        np.sum(V)
    )

    return (
        kinetic
        + potential_linear
        + potential_nonlinear
    )


# =============================================================================
# NORMA L2
# =============================================================================

def compute_l2_norm(gamma):

    return np.sqrt(
        np.sum(gamma**2)
    )


# =============================================================================
# AMPLITUDE MÁXIMA
# =============================================================================

def compute_max_amplitude(gamma):

    return np.max(
        np.abs(gamma)
    )


# =============================================================================
# PROJEÇÃO ESPECTRAL
# =============================================================================

def modal_projection(
    gamma,
    eigenvectors
):
    """
    Projeta o campo na base modal.
    """

    modal = (
        eigenvectors.T
        @ gamma
    )

    energy = modal**2

    probability = (
        energy /
        (
            np.sum(energy)
            + EPS
        )
    )

    return (
        modal,
        energy,
        probability
    )


# =============================================================================
# ENTROPIA ESPECTRAL
# =============================================================================

def compute_spectral_entropy(
    probability
):

    return -np.sum(
        probability *
        np.log(
            probability + EPS
        )
    )


# =============================================================================
# MODO DOMINANTE
# =============================================================================

def dominant_mode(
    probability
):

    return int(
        np.argmax(
            probability
        )
    )


# =============================================================================
# CENTRO MODAL
# =============================================================================

def modal_center(
    probability
):

    modes = np.arange(
        len(probability)
    )

    return np.sum(
        modes *
        probability
    )


# =============================================================================
# LARGURA MODAL
# =============================================================================

def modal_width(
    probability
):

    modes = np.arange(
        len(probability)
    )

    center = modal_center(
        probability
    )

    variance = np.sum(
        probability *
        (modes-center)**2
    )

    return np.sqrt(
        variance
    )


# =============================================================================
# MÉTRICAS CIRCULARES
# =============================================================================

def compute_circular_metrics(
    gamma,
    theta
):
    """
    Estatística circular consistente
    para topologia periódica.
    """

    weight = np.abs(gamma)

    norm = np.sum(weight)

    if norm < EPS:

        return {
            "center":0.0,
            "width":0.0,
            "R":0.0,
            "skewness":0.0
        }

    weight = weight / norm

    C = np.sum(
        weight *
        np.cos(theta)
    )

    S = np.sum(
        weight *
        np.sin(theta)
    )

    R = np.sqrt(
        C**2 + S**2
    )

    theta_mean = np.arctan2(
        S,
        C
    )

    theta_dev = np.arctan2(
        np.sin(theta-theta_mean),
        np.cos(theta-theta_mean)
    )

    width = np.sqrt(
        max(
            0.0,
            -2.0*np.log(
                max(R,EPS)
            )
        )
    )

    skewness = np.sum(
        weight *
        np.sin(
            3.0*theta_dev
        )
    )

    return {

        "center":theta_mean,

        "width":width,

        "R":R,

        "skewness":skewness
    }


# =============================================================================
# ERRO RELATIVO DE ENERGIA
# =============================================================================

def relative_energy_error(
    energy_initial,
    energy_final
):

    return (
        np.abs(
            energy_final
            -
            energy_initial
        )
        /
        (
            np.abs(
                energy_initial
            )
            +
            EPS
        )
    )


# =============================================================================
# SNAPSHOT PADRONIZADO
# =============================================================================

def build_snapshot(
    step,
    time,
    gamma,
    velocity,
    L,
    beta,
    potential,
    eigenvectors,
    theta
):
    """
    Gera um snapshot completo do estado.
    """

    H = compute_hamiltonian(
        gamma,
        velocity,
        L,
        beta,
        potential
    )

    l2 = compute_l2_norm(
        gamma
    )

    amp = compute_max_amplitude(
        gamma
    )

    (
    modal,
    modal_energy,
    probability,
) = modal_projection(
    gamma,
    eigenvectors
)
    entropy = compute_spectral_entropy(
        probability
    )

    circle = compute_circular_metrics(
        gamma,
        theta
    )

    return {

        "step":step,

        "time":time,

        "energy":H,

        "l2":l2,

        "amplitude":amp,

        "dominant_mode":
            dominant_mode(
                probability
            ),

        "modal_center":
            modal_center(
                probability
            ),

        "modal_width":
            modal_width(
                probability
            ),

        "spectral_entropy":
            entropy,

        "circular":
            circle,

        "probability":
            probability,

        "modal_energy":
            modal_energy
    }
# =============================================================================
# MOTOR DE EVOLUÇÃO TEMPORAL
# =============================================================================

def run_engine(
    n=384,
    timesteps=2000,
    dt=2.5e-4,
    beta=1.0,
    potential="A",
    snapshot_stride=50,
    sigma=0.10
):
    """
    Motor único da série S26-B.

    Retorna toda a evolução necessária para qualquer auditoria.
    """

    _, L, theta = build_ring_graph(n)

    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(
        theta,
        sigma=sigma
    )

    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )

    velocity0 = (gamma - gamma_old) / dt

    energy0 = compute_hamiltonian(
        gamma,
        velocity0,
        L,
        beta,
        potential
    )

    snapshots = []

    diverged = False

    for step in range(timesteps):

        force, _ = Potential.evaluate(
            gamma,
            potential
        )

        acceleration = (
            -(L @ gamma)
            +
            beta*force
        )

        gamma_new = (
            2.0*gamma
            -
            gamma_old
            +
            dt*dt*acceleration
        )

        velocity = (
            gamma_new
            -
            gamma_old
        ) / (2.0*dt)

        energy = compute_hamiltonian(
            gamma,
            velocity,
            L,
            beta,
            potential
        )

        error = relative_energy_error(
            energy0,
            energy
        )

        if check_divergence(
            gamma,
            error
        ):
            diverged = True

        if (
            step % snapshot_stride == 0
            or step == timesteps-1
        ):

            snap = build_snapshot(
                step=step,
                time=step*dt,
                gamma=gamma,
                velocity=velocity,
                L=L,
                beta=beta,
                potential=potential,
                eigenvectors=eigenvectors,
                theta=theta
            )

            snap["energy_error"] = error

            snapshots.append(snap)

        gamma_old = gamma
        gamma = gamma_new

    final_velocity = (
        gamma-gamma_old
    )/dt

    final_energy = compute_hamiltonian(
        gamma,
        final_velocity,
        L,
        beta,
        potential
    )

    final_error = relative_energy_error(
        energy0,
        final_energy
    )

    return {

        "configuration":{

            "n":n,

            "dt":dt,

            "timesteps":timesteps,

            "beta":beta,

            "potential":potential

        },

        "initial":{

            "energy":energy0,

            "l2":compute_l2_norm(gamma_old),

            "amplitude":compute_max_amplitude(gamma_old)

        },

        "final":{

            "energy":final_energy,

            "l2":compute_l2_norm(gamma),

            "amplitude":compute_max_amplitude(gamma),

            "energy_error":final_error

        },

        "snapshots":snapshots,

        "gamma":gamma,

        "laplacian":L,

        "eigenvalues":eigenvalues,

        "eigenvectors":eigenvectors,

        "diverged":diverged

    }


# =============================================================================
# EXECUTOR DE MATRIZ DE PARÂMETROS
# =============================================================================

def run_parameter_grid(
    beta_values,
    potentials,
    **kwargs
):
    """
    Executa automaticamente todas
    as combinações Potencial × Beta.
    """

    results = {}

    for pot in potentials:

        results[pot] = {}

        for beta in beta_values:

            results[pot][beta] = run_engine(
                beta=beta,
                potential=pot,
                **kwargs
            )

    return results


# =============================================================================
# RESUMO NUMÉRICO
# =============================================================================

def summarize_run(result):

    return {

        "energy_error":
            result["final"]["energy_error"],

        "final_l2":
            result["final"]["l2"],

        "final_amplitude":
            result["final"]["amplitude"],

        "diverged":
            result["diverged"]

    }


# =============================================================================
# FIM DO NÚCLEO
# =============================================================================

Overwriting ger_engine.py


In [23]:
print(run_engine)

<function run_engine at 0x7a29fd6c3240>


In [24]:
resultado = run_engine(
    beta=1.0,
    potential="A"
)

print(resultado.keys())

dict_keys(['configuration', 'initial', 'final', 'snapshots', 'gamma', 'laplacian', 'eigenvalues', 'eigenvectors', 'diverged'])


In [25]:
print(resultado["initial"])
print(resultado["final"])
print(resultado["diverged"])
print(len(resultado["snapshots"]))

{'energy': np.float64(-2.8759545182212616), 'l2': np.float64(3.634409638230734), 'amplitude': np.float64(1.1266911038069758)}
{'energy': np.float64(-2.876569170820299), 'l2': np.float64(3.6347665857511147), 'amplitude': np.float64(1.1268232534751075), 'energy_error': np.float64(0.00021372125155085715)}
False
41


In [26]:
print(resultado["snapshots"][0].keys())

dict_keys(['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'circular', 'probability', 'modal_energy', 'energy_error'])


In [27]:
import numpy as np


# ============================================================
# S26-B.2.2
# AUDITORIA DE TRANSFERÊNCIA MODAL
# ============================================================


def analyze_modal_transfer(result):
    """
    Analisa evolução espectral usando snapshots
    produzidos pelo ger_engine.

    Entrada:
        result = saída do run_engine()

    Retorno:
        dicionário com séries temporais
    """

    snapshots = result["snapshots"]

    times = []
    entropy = []
    spectral_center = []
    spectral_width = []
    dominant_modes = []

    low_energy = []
    mid_energy = []
    high_energy = []

    for snap in snapshots:

        p = np.array(
            snap["probability"],
            dtype=float
        )

        p = p / (np.sum(p) + 1e-15)

        modes = np.arange(len(p))


        # ---------------------------------
        # Entropia espectral
        # ---------------------------------

        S = -np.sum(
            p * np.log(p + 1e-15)
        )


        # ---------------------------------
        # Centro espectral
        # ---------------------------------

        kc = np.sum(
            modes * p
        )


        # ---------------------------------
        # Largura espectral
        # ---------------------------------

        width = np.sqrt(
            np.sum(
                (modes-kc)**2 * p
            )
        )


        # ---------------------------------
        # Faixas espectrais
        # ---------------------------------

        n = len(p)

        i1 = int(0.25*n)
        i2 = int(0.75*n)


        low = np.sum(
            p[:i1]
        )

        mid = np.sum(
            p[i1:i2]
        )

        high = np.sum(
            p[i2:]
        )


        # guardar

        times.append(
            snap["time"]
        )

        entropy.append(S)

        spectral_center.append(kc)

        spectral_width.append(width)

        dominant_modes.append(
            snap["dominant_mode"]
        )

        low_energy.append(low)

        mid_energy.append(mid)

        high_energy.append(high)



    return {

        "time": np.array(times),

        "entropy": np.array(entropy),

        "spectral_center": np.array(
            spectral_center
        ),

        "spectral_width": np.array(
            spectral_width
        ),

        "dominant_mode": np.array(
            dominant_modes
        ),

        "low_energy": np.array(
            low_energy
        ),

        "mid_energy": np.array(
            mid_energy
        ),

        "high_energy": np.array(
            high_energy
        )

    }



# ============================================================
# TAXAS DE TRANSFERÊNCIA
# ============================================================


def calculate_transfer_rates(data):

    """
    Calcula derivadas temporais discretas
    das grandezas espectrais.
    """

    t = data["time"]

    dt = np.diff(t)


    rates = {}


    for key in [

        "entropy",
        "spectral_center",
        "spectral_width",
        "high_energy"

    ]:

        values = data[key]


        rates[key+"_rate"] = (
            np.diff(values)
            /
            (dt + 1e-15)
        )


    return rates



# ============================================================
# RELATÓRIO
# ============================================================


def print_modal_report(data, rates):

    print("="*70)
    print("S26-B.2.2 — RELATÓRIO DE TRANSFERÊNCIA MODAL")
    print("="*70)


    print()

    print(
        "Entropia inicial/final:",
        data["entropy"][0],
        " --> ",
        data["entropy"][-1]
    )


    print(
        "Centro espectral:",
        data["spectral_center"][0],
        " --> ",
        data["spectral_center"][-1]
    )


    print(
        "Largura espectral:",
        data["spectral_width"][0],
        " --> ",
        data["spectral_width"][-1]
    )


    print(
        "Modo dominante:",
        data["dominant_mode"][0],
        " --> ",
        data["dominant_mode"][-1]
    )


    print()

    print(
        "Energia baixa frequência:",
        data["low_energy"][0],
        " --> ",
        data["low_energy"][-1]
    )


    print(
        "Energia média frequência:",
        data["mid_energy"][0],
        " --> ",
        data["mid_energy"][-1]
    )


    print(
        "Energia alta frequência:",
        data["high_energy"][0],
        " --> ",
        data["high_energy"][-1]
    )


    print()

    print(
        "Taxa média crescimento entropia:",
        np.mean(
            rates["entropy_rate"]
        )
    )


    print(
        "Taxa média transferência alta frequência:",
        np.mean(
            rates["high_energy_rate"]
        )
    )


    print("="*70)



# ============================================================
# EXECUÇÃO DIRETA
# ============================================================


def run_B22_case(engine_result):

    data = analyze_modal_transfer(
        engine_result
    )

    rates = calculate_transfer_rates(
        data
    )

    print_modal_report(
        data,
        rates
    )


    return {

        "analysis": data,

        "rates": rates

    }

In [28]:
print(run_B22_case)

<function run_B22_case at 0x7a2a18e07920>


In [29]:
print(run_engine)

<function run_engine at 0x7a29fd6c3240>


In [30]:
from ger_engine import run_engine
from S26_B2_2_modal_audit import run_B22_case

ModuleNotFoundError: No module named 'S26_B2_2_modal_audit'

In [17]:
from ger_engine import run_engine

In [19]:
import inspect

print(inspect.getsource(run_engine))

def run_engine(
    n=384,
    timesteps=2000,
    dt=2.5e-4,
    beta=1.0,
    potential="A",
    snapshot_stride=50,
    sigma=0.10
):
    """
    Motor único da série S26-B.

    Retorna toda a evolução necessária para qualquer auditoria.
    """

    _, L, theta = build_ring_graph(n)

    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(
        theta,
        sigma=sigma
    )

    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )

    velocity0 = (gamma - gamma_old) / dt

    energy0 = compute_hamiltonian(
        gamma,
        velocity0,
        L,
        beta,
        potential
    )

    snapshots = []

    diverged = False

    for step in range(timesteps):

        force, _ = Potential.evaluate(
            gamma,
            potential
        )

        acceleration = (
            -(L @ gamma)
            +
            beta*force
        )

        gamma_new = (
            2.0*gamma
     

In [20]:
%%writefile ger_engine.py

"""
GER Engine v0.1
S26-B.2 Base Engine
"""

import numpy as np
import scipy.linalg as la


# funções do motor aqui

Overwriting ger_engine.py


In [34]:
%%writefile ger_engine.py
import numpy as np
import scipy.linalg as la

class Potential:
    @staticmethod
    def evaluate(gamma, mode="A"):
        # Mapeia tanto o nome descritivo quanto a letra para o potencial correto
        if mode in ["A", "Gamma^2"]:
            return gamma**2, (1.0 / 3.0) * gamma**3
        elif mode in ["B", "Saturado"]:
            return gamma**2 / (1.0 + gamma**2), gamma - np.arctan(gamma)
        elif mode in ["C", "Gamma^3"]:
            return gamma**3, 0.25 * gamma**4
        else:
            raise ValueError(f"Potencial desconhecido: {mode}")

def build_ring_graph(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = 1
        A[i, (i - 1) % n] = 1
    L = np.diag(np.sum(A, axis=1)) - A
    return A, L

def spectral_basis(L):
    eigenvalues, eigenvectors = la.eigh(L)
    return eigenvalues, eigenvectors

def gaussian_packet(n, sigma=4.0):
    x = np.arange(n)
    center = n // 2
    gamma = np.exp(-(x - center)**2 / (2 * sigma**2))
    return gamma

def compute_hamiltonian(gamma, vel, L, beta, potential_mode):
    kinetic = 0.5 * np.sum(vel**2)
    potential_linear = 0.5 * np.dot(gamma, L @ gamma)
    _, V = Potential.evaluate(gamma, potential_mode)
    potential_nonlinear = -beta * np.sum(V)
    return kinetic + potential_linear + potential_nonlinear

def run_engine(n=768, timesteps=1000, dt=1.25e-4, beta=0.5, potential="A", snapshot_stride=100):
    A, L = build_ring_graph(n)
    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(n)
    gamma_old = gamma.copy()

    modal_history = []
    snapshots = []

    for step in range(timesteps):
        t = step * dt
        F, _ = Potential.evaluate(gamma, potential)

        accel = -1.0 * (L @ gamma) + beta * F
        gamma_new = 2 * gamma - gamma_old + dt**2 * accel

        # Projeção espectral para registrar o histórico modal exigido na série B2
        modal_amplitudes = np.dot(gamma, eigenvectors)
        modal_history.append(modal_amplitudes**2)

        if step % snapshot_stride == 0:
            snapshots.append({"step": step, "t": t, "gamma": gamma.copy()})

        gamma_old = gamma
        gamma = gamma_new

    resultados = {
        "status": "CONCLUIDO",
        "passos_totais": timesteps,
        "chaves_validas": ["config", "metricas"]
    }

    return resultados, snapshots, np.array(modal_history)

Overwriting ger_engine.py


In [35]:
# Força o Python a recarregar o arquivo ger_engine modificado
import sys
import importlib
if 'ger_engine' in sys.modules:
    importlib.reload(sys.modules['ger_engine'])

from ger_engine import run_engine

print("Acionando o motor empacotado com Potencial A...")
resultados, snapshots, modal_hist = run_engine(beta=1.0, potential="A")

print("\nChaves do resultado obtido:")
print(resultados.keys())
print(f"Passos registrados no histórico modal: {len(modal_hist)}")
print("\nSucesso total! O motor aceitou o potencial e concluiu a simulação sem NameError ou ValueError.")

Acionando o motor empacotado com Potencial A...

Chaves do resultado obtido:
dict_keys(['status', 'passos_totais', 'chaves_validas'])
Passos registrados no histórico modal: 1000

Sucesso total! O motor aceitou o potencial e concluiu a simulação sem NameError ou ValueError.


In [36]:
# test_engine.py

from ger_engine import run_engine

resultado = run_engine(
    beta=1.0,
    potential="A"
)

assert resultado["status"] == "success"

print("Motor GER S26-B validado")

TypeError: tuple indices must be integers or slices, not str

In [37]:
print(resultado.keys())

AttributeError: 'tuple' object has no attribute 'keys'

In [38]:
print(resultado["chaves_validas"])

TypeError: tuple indices must be integers or slices, not str

In [39]:
print(type(resultado))
print(len(resultado))
print(resultado)

<class 'tuple'>
3
({'status': 'CONCLUIDO', 'passos_totais': 1000, 'chaves_validas': ['config', 'metricas']}, [{'step': 0, 't': 0.0, 'gamma': array([0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000

In [40]:
print(type(resultado))
print(len(resultado))

for i, item in enumerate(resultado):
    print("Índice", i, "tipo:", type(item))

<class 'tuple'>
3
Índice 0 tipo: <class 'dict'>
Índice 1 tipo: <class 'list'>
Índice 2 tipo: <class 'numpy.ndarray'>


In [41]:
dados = resultado[0]

print(dados.keys())

dict_keys(['status', 'passos_totais', 'chaves_validas'])


In [42]:
historico = resultado[1]

print("Quantidade de snapshots:", len(historico))

print(historico[0].keys())

Quantidade de snapshots: 10
dict_keys(['step', 't', 'gamma'])


In [43]:
campo = resultado[2]

print(campo.shape)

(1000, 768)


In [44]:
def normalize_engine_output(resultado):
    return {
        "configuration": resultado[0],
        "snapshots": resultado[1],
        "gamma": resultado[2]
    }

In [45]:
print(resultado[1][-1].keys())

dict_keys(['step', 't', 'gamma'])


In [46]:
%%writefile ger_engine.py
import numpy as np
import scipy.linalg as la

class Potential:
    @staticmethod
    def evaluate(gamma, mode="A"):
        if mode in ["A", "Gamma^2"]:
            return gamma**2, (1.0 / 3.0) * gamma**3
        elif mode in ["B", "Saturado"]:
            return gamma**2 / (1.0 + gamma**2), gamma - np.arctan(gamma)
        elif mode in ["C", "Gamma^3"]:
            return gamma**3, 0.25 * gamma**4
        else:
            raise ValueError(f"Potencial desconhecido: {mode}")

def build_ring_graph(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = 1
        A[i, (i - 1) % n] = 1
    L = np.diag(np.sum(A, axis=1)) - A
    return A, L

def spectral_basis(L):
    eigenvalues, eigenvectors = la.eigh(L)
    return eigenvalues, eigenvectors

def gaussian_packet(n, sigma=4.0):
    x = np.arange(n)
    center = n // 2
    gamma = np.exp(-(x - center)**2 / (2 * sigma**2))
    return gamma

def compute_hamiltonian(gamma, vel, L, beta, potential_mode):
    kinetic = 0.5 * np.sum(vel**2)
    potential_linear = 0.5 * np.dot(gamma, L @ gamma)
    _, V = Potential.evaluate(gamma, potential_mode)
    potential_nonlinear = -beta * np.sum(V)
    return kinetic + potential_linear + potential_nonlinear

def build_snapshot_metrics(step, t, gamma, vel, L, beta, potential, eigenvectors, E0):
    # Cálculo de Energia Absoluta e Erro Hamiltoniano
    H = compute_hamiltonian(gamma, vel, L, beta, potential)
    energy_error = abs(H - E0) / abs(E0) if abs(E0) > 1e-15 else abs(H - E0)

    # Métricas no Espaço Real
    l2 = np.sqrt(np.sum(gamma**2))
    amplitude = np.max(np.abs(gamma))

    # Projeção Espectral Completa (Auditoria de Modos)
    modal = eigenvectors.T @ gamma
    modal_energy = modal**2
    probability = modal_energy / (np.sum(modal_energy) + 1e-15)

    # Análise Estatística Espectral
    dominant_mode = int(np.argmax(modal_energy))
    spectral_entropy = -np.sum(probability * np.log(probability + 1e-15))

    # Centro e Largura Espectral
    k_indices = np.arange(len(modal_energy))
    modal_center = np.sum(k_indices * probability)
    modal_width = np.sqrt(np.sum((k_indices - modal_center)**2 * probability) + 1e-15)

    return {
        "step": step,
        "time": t,
        "energy": H,
        "l2": l2,
        "amplitude": amplitude,
        "dominant_mode": dominant_mode,
        "modal_center": modal_center,
        "modal_width": modal_width,
        "spectral_entropy": spectral_entropy,
        "probability": probability,
        "modal_energy": modal_energy,
        "energy_error": energy_error
    }

def run_engine(n=768, timesteps=1000, dt=1.25e-4, beta=0.5, potential="A", snapshot_stride=100):
    A, L = build_ring_graph(n)
    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(n)
    gamma_old = gamma.copy()

    # Inicialização de segunda ordem estável para velocidade inicial
    F0, _ = Potential.evaluate(gamma, potential)
    accel0 = -1.0 * (L @ gamma) + beta * F0
    vel0 = dt * accel0 / 2.0  # Aproximação de partida de Verlet

    E0 = compute_hamiltonian(gamma, vel0, L, beta, potential)

    snapshots_list = []

    # Captura do Estado Inicial Mapeado
    initial_metrics = build_snapshot_metrics(0, 0.0, gamma, vel0, L, beta, potential, eigenvectors, E0)
    snapshots_list.append(initial_metrics)

    for step in range(1, timesteps):
        t = step * dt
        F, _ = Potential.evaluate(gamma, potential)

        accel = -1.0 * (L @ gamma) + beta * F
        gamma_new = 2 * gamma - gamma_old + dt**2 * accel

        # Velocidade centralizada para cálculo preciso da energia
        vel = (gamma_new - gamma_old) / (2.0 * dt)

        if step % snapshot_stride == 0:
            metrics = build_snapshot_metrics(step, t, gamma, vel, L, beta, potential, eigenvectors, E0)
            snapshots_list.append(metrics)

        gamma_old = gamma
        gamma = gamma_new

    # Captura do Estado Final Mapeado
    vel_final = (gamma - gamma_old) / dt
    final_metrics = build_snapshot_metrics(timesteps, timesteps * dt, gamma, vel_final, L, beta, potential, eigenvectors, E0)

    # Estruturação do Dicionário de Saída Rich exigido pela série B2
    resultado_rich = {
        "configuration": {"n": n, "timesteps": timesteps, "dt": dt, "beta": beta, "potential": potential},
        "initial": initial_metrics,
        "final": final_metrics,
        "snapshots": snapshots_list,
        "gamma": gamma.copy(),
        "laplacian": L,
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "diverged": bool(final_metrics["energy_error"] > 0.5)
    }

    return resultado_rich

Overwriting ger_engine.py


In [47]:
import sys
import importlib
if 'ger_engine' in sys.modules:
    importlib.reload(sys.modules['ger_engine'])

from ger_engine import run_engine

print("Acionando o motor v0.2 com Auditoria Espectral Avançada...")
resultado = run_engine(beta=1.0, potential="A")

print("\n--- VALIDAÇÃO DO MOTOR ---")
print("Chaves principais do resultado:")
print(list(resultado.keys()))

print("\nChaves internas de um snapshot amostrado:")
print(list(resultado["snapshots"][0].keys()))

print(f"\nStatus de divergência: {resultado['diverged']}")
print("Sucesso! O ecossistema espectral foi restaurado.")

Acionando o motor v0.2 com Auditoria Espectral Avançada...

--- VALIDAÇÃO DO MOTOR ---
Chaves principais do resultado:
['configuration', 'initial', 'final', 'snapshots', 'gamma', 'laplacian', 'eigenvalues', 'eigenvectors', 'diverged']

Chaves internas de um snapshot amostrado:
['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'probability', 'modal_energy', 'energy_error']

Status de divergência: False
Sucesso! O ecossistema espectral foi restaurado.


In [48]:
print(resultado[1][-1].keys())

KeyError: 1

In [49]:
resultado_A = run_engine(
    beta=1.0,
    potential="A"
)

In [50]:
analysis_B21 = run_B21_case(resultado_A)

print(analysis_B21.keys())

NameError: name 'run_B21_case' is not defined

In [51]:
analysis_B22 = run_B22_case(resultado_A)

print(analysis_B22.keys())

S26-B.2.2 — RELATÓRIO DE TRANSFERÊNCIA MODAL

Entropia inicial/final: 3.8116336866228786  -->  3.8129512159899623
Centro espectral: 33.771984665554356  -->  33.81368851467764
Largura espectral: 26.11845981258404  -->  26.158915657344004
Modo dominante: 1  -->  1

Energia baixa frequência: 0.9999920461152646  -->  0.9999910232422369
Energia média frequência: 7.953884734354885e-06  -->  8.97675776182701e-06
Energia alta frequência: 2.0785625180402746e-30  -->  2.7237795945154596e-24

Taxa média crescimento entropia: 0.011711372151854497
Taxa média transferência alta frequência: 2.4211355697357548e-23
dict_keys(['analysis', 'rates'])


In [52]:
import numpy as np


def count_local_peaks(signal):
    """
    Conta máximos locais simples.
    """
    peaks = 0

    for i in range(1, len(signal)-1):
        if (
            abs(signal[i]) > abs(signal[i-1])
            and abs(signal[i]) > abs(signal[i+1])
        ):
            peaks += 1

    return peaks



def participation_ratio(probability):
    """
    Mede quantos modos participam efetivamente.

    P = 1 / sum(p_k^2)
    """
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )



def spatial_localization_metrics(snapshot):
    """
    Extrai métricas espaciais de um snapshot.
    """

    gamma = snapshot["gamma"]

    abs_gamma = np.abs(gamma)

    # amplitude máxima
    amplitude = np.max(abs_gamma)

    # norma
    l2 = np.sqrt(
        np.sum(gamma**2)
    )

    # índice do pico
    peak_position = np.argmax(abs_gamma)

    # número de estruturas locais
    peaks = count_local_peaks(gamma)

    # participação modal
    if "probability" in snapshot:
        participation = participation_ratio(
            snapshot["probability"]
        )
    else:
        participation = None


    return {
        "amplitude": amplitude,
        "l2": l2,
        "peak_position": peak_position,
        "local_peaks": peaks,
        "participation_ratio": participation
    }



def run_S26_B23_analysis(resultado):

    print("="*80)
    print("S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL")
    print("="*80)


    history = resultado["snapshots"]


    analysis = []


    for snap in history:

        metrics = spatial_localization_metrics(
            snap
        )

        metrics["step"] = snap["step"]
        metrics["time"] = snap["time"]

        analysis.append(metrics)


    initial = analysis[0]
    final = analysis[-1]


    print("\nESTADO INICIAL")
    print("----------------")
    for k,v in initial.items():
        print(k,":",v)


    print("\nESTADO FINAL")
    print("----------------")
    for k,v in final.items():
        print(k,":",v)



    print("\nVARIAÇÕES")
    print("----------------")

    print(
        "Amplitude:",
        initial["amplitude"],
        " --> ",
        final["amplitude"]
    )

    print(
        "L2:",
        initial["l2"],
        " --> ",
        final["l2"]
    )

    print(
        "Picos locais:",
        initial["local_peaks"],
        " --> ",
        final["local_peaks"]
    )

    print(
        "Participação modal:",
        initial["participation_ratio"],
        " --> ",
        final["participation_ratio"]
    )


    return {
        "initial":initial,
        "final":final,
        "history":analysis
    }

In [53]:
analysis_B23 = run_S26_B23_analysis(resultado)

S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL


KeyError: 'gamma'

In [54]:
import numpy as np


def participation_ratio(probability):
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )


def run_S26_B23_analysis(resultado):

    print("="*80)
    print("S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL")
    print("="*80)


    snapshots = resultado["snapshots"]

    history = []


    for snap in snapshots:

        if "probability" in snap:
            participation = participation_ratio(
                snap["probability"]
            )
        else:
            participation = None


        metrics = {

            "step": snap["step"],

            "time": snap["time"],

            "energy": snap["energy"],

            "l2": snap["l2"],

            "amplitude": snap["amplitude"],

            "dominant_mode": snap["dominant_mode"],

            "spectral_entropy": snap["spectral_entropy"],

            "modal_width": snap["modal_width"],

            "participation_ratio": participation
        }


        history.append(metrics)



    initial = history[0]
    final = history[-1]


    print("\nESTADO INICIAL")
    print("----------------")

    for k,v in initial.items():
        print(k,":",v)



    print("\nESTADO FINAL")
    print("----------------")

    for k,v in final.items():
        print(k,":",v)



    print("\nVARIAÇÕES")
    print("----------------")

    print(
        "Amplitude:",
        initial["amplitude"],
        "-->",
        final["amplitude"]
    )

    print(
        "L2:",
        initial["l2"],
        "-->",
        final["l2"]
    )

    print(
        "Entropia espectral:",
        initial["spectral_entropy"],
        "-->",
        final["spectral_entropy"]
    )

    print(
        "Largura modal:",
        initial["modal_width"],
        "-->",
        final["modal_width"]
    )

    print(
        "Participação modal:",
        initial["participation_ratio"],
        "-->",
        final["participation_ratio"]
    )


    return {

        "initial": initial,

        "final": final,

        "history": history

    }

In [55]:
analysis_B23 = run_S26_B23_analysis(resultado)

S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL

ESTADO INICIAL
----------------
step : 0
time : 0.0
energy : -1.8196859368600302
l2 : 2.6626707276007795
amplitude : 1.0
dominant_mode : 1
spectral_entropy : 3.8116336866228813
modal_width : 26.118459812584053
participation_ratio : 38.805072944798845

ESTADO FINAL
----------------
step : 900
time : 0.1125
energy : -1.8196859366647562
l2 : 2.675916036561578
amplitude : 1.005943924833755
dominant_mode : 1
spectral_entropy : 3.8129512159899646
modal_width : 26.158915657344018
participation_ratio : 38.84890013659818

VARIAÇÕES
----------------
Amplitude: 1.0 --> 1.005943924833755
L2: 2.6626707276007795 --> 2.675916036561578
Entropia espectral: 3.8116336866228813 --> 3.8129512159899646
Largura modal: 26.118459812584053 --> 26.158915657344018
Participação modal: 38.805072944798845 --> 38.84890013659818


In [56]:
import numpy as np


def modal_participation(probability):
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )


def run_S26_B24_scan(
    beta_values=None,
    potential="A",
    dt=2.5e-4,
    timesteps=2000
):

    if beta_values is None:
        beta_values = [
            0.0,
            0.25,
            0.5,
            0.75,
            1.0,
            1.25,
            1.5,
            2.0
        ]


    print("="*90)
    print("S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO")
    print("="*90)

    results = {}


    for beta in beta_values:

        print("\nExecutando beta =", beta)


        resultado = run_engine(
            beta=beta,
            potential=potential,
            dt=dt,
            timesteps=timesteps
        )


        snap_i = resultado["snapshots"][0]
        snap_f = resultado["snapshots"][-1]


        participation_i = modal_participation(
            snap_i["probability"]
        )

        participation_f = modal_participation(
            snap_f["probability"]
        )


        data = {

            "beta": beta,

            "diverged": resultado["diverged"],

            "energy_initial":
                snap_i["energy"],

            "energy_final":
                snap_f["energy"],

            "energy_error":
                snap_f["energy_error"],

            "amplitude_initial":
                snap_i["amplitude"],

            "amplitude_final":
                snap_f["amplitude"],

            "l2_initial":
                snap_i["l2"],

            "l2_final":
                snap_f["l2"],

            "entropy_initial":
                snap_i["spectral_entropy"],

            "entropy_final":
                snap_f["spectral_entropy"],

            "width_initial":
                snap_i["modal_width"],

            "width_final":
                snap_f["modal_width"],

            "participation_initial":
                participation_i,

            "participation_final":
                participation_f,

            "mode_initial":
                snap_i["dominant_mode"],

            "mode_final":
                snap_f["dominant_mode"]

        }


        results[beta] = data


    return results

In [57]:
resultado_B24_A = run_S26_B24_scan(
    potential="A"
)

S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO

Executando beta = 0.0

Executando beta = 0.25

Executando beta = 0.5

Executando beta = 0.75

Executando beta = 1.0

Executando beta = 1.25

Executando beta = 1.5

Executando beta = 2.0


In [58]:
resultado_B24_C = run_S26_B24_scan(
    potential="C"
)

S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO

Executando beta = 0.0

Executando beta = 0.25

Executando beta = 0.5

Executando beta = 0.75

Executando beta = 1.0

Executando beta = 1.25

Executando beta = 1.5

Executando beta = 2.0


In [59]:
def inspect_B24(results):

    print("="*90)
    print("INSPEÇÃO S26-B.2.4")
    print("="*90)

    for beta,data in results.items():

        print("\nβ =", beta)

        print(
            "Erro energia:",
            data["energy_error"]
        )

        print(
            "Amplitude:",
            data["amplitude_initial"],
            " --> ",
            data["amplitude_final"]
        )

        print(
            "Largura modal:",
            data["width_initial"],
            " --> ",
            data["width_final"]
        )

        print(
            "Entropia:",
            data["entropy_initial"],
            " --> ",
            data["entropy_final"]
        )

        print(
            "Participação:",
            data["participation_initial"],
            " --> ",
            data["participation_final"]
        )

        print(
            "Modo:",
            data["mode_initial"],
            " --> ",
            data["mode_final"]
        )

        print(
            "Divergiu:",
            data["diverged"]
        )

In [60]:
inspect_B24(resultado_B24_A)

INSPEÇÃO S26-B.2.4

β = 0.0
Erro energia: 4.820915626631271e-11
Amplitude: 1.0  -->  0.9930854285316857
Largura modal: 26.118459812584053  -->  25.93830429457281
Entropia: 3.8116336866228813  -->  3.804784463430525
Participação: 38.805072944798845  -->  38.53974782780961
Modo: 1  -->  1
Divergiu: False

β = 0.25
Erro energia: 4.0401605432659316e-10
Amplitude: 1.0  -->  1.0214105733721073
Largura modal: 26.118459812584053  -->  26.164448636730654
Entropia: 3.8116336866228813  -->  3.812405936224194
Participação: 38.805072944798845  -->  38.80181830145292
Modo: 1  -->  1
Divergiu: False

β = 0.5
Erro energia: 1.9420989233654438e-09
Amplitude: 1.0  -->  1.050277410774106
Largura modal: 26.118459812584053  -->  26.39025499910629
Entropia: 3.8116336866228813  -->  3.8199419636940584
Participação: 38.805072944798845  -->  39.06293834785101
Modo: 1  -->  1
Divergiu: False

β = 0.75
Erro energia: 4.880224061330748e-09
Amplitude: 1.0  -->  1.079701513549304
Largura modal: 26.118459812584053  --

In [5]:
%%writefile GER_CORE/ger_graph.py

"""
=========================================================
GER CORE
Arquivo : ger_graph.py
=========================================================
...
"""
=========================================================
GER CORE
Arquivo : ger_graph.py
=========================================================

Geometria discreta da Geometria Espectral Relacional.

Este módulo implementa:

• Grafo periódico F₁
• Laplaciano discreto
• Base espectral
• Distância periódica
• Pacotes gaussianos iniciais

Todas as funções aqui são independentes da dinâmica.
"""

from __future__ import annotations

import numpy as np
import scipy.linalg as la


# =========================================================
# Construção do Grafo
# =========================================================

def build_ring_graph(n: int):
    """
    Constrói o grafo periódico F₁.

    Parameters
    ----------
    n : int
        Número de vértices.

    Returns
    -------
    adjacency : ndarray
        Matriz de adjacência.

    laplacian : ndarray
        Laplaciano discreto.

    theta : ndarray
        Coordenadas angulares dos vértices.
    """

    if n < 3:
        raise ValueError(
            "O grafo periódico deve possuir pelo menos 3 vértices."
        )

    adjacency = np.zeros((n, n))

    for i in range(n):

        adjacency[i, (i + 1) % n] = 1.0
        adjacency[i, (i - 1) % n] = 1.0

    degree = np.diag(np.sum(adjacency, axis=1))

    laplacian = degree - adjacency

    theta = np.linspace(
        0.0,
        2.0 * np.pi,
        n,
        endpoint=False
    )

    return adjacency, laplacian, theta


# =========================================================
# Base Espectral
# =========================================================

def spectral_basis(laplacian):
    """
    Calcula a base espectral do Laplaciano.

    Parameters
    ----------
    laplacian : ndarray

    Returns
    -------
    eigenvalues : ndarray

    eigenvectors : ndarray
    """

    eigenvalues, eigenvectors = la.eigh(laplacian)

    eigenvalues[np.abs(eigenvalues) < 1e-12] = 0.0

    return eigenvalues, eigenvectors


# =========================================================
# Distância Periódica
# =========================================================

def periodic_distance(theta, center):
    """
    Distância angular mínima em uma circunferência.

    Parameters
    ----------
    theta : ndarray

    center : float

    Returns
    -------
    ndarray
    """

    delta = np.abs(theta - center)

    return np.minimum(
        delta,
        2.0 * np.pi - delta
    )


# =========================================================
# Pacote Inicial
# =========================================================

def gaussian_packet(
    theta,
    center=np.pi,
    sigma=0.10
):
    """
    Constrói um pacote gaussiano periódico.

    Parameters
    ----------
    theta : ndarray

    center : float

    sigma : float

    Returns
    -------
    gamma : ndarray
    """

    distance = periodic_distance(
        theta,
        center
    )

    gamma = np.exp(
        -(distance ** 2) /
        (2.0 * sigma ** 2)
    )

    gamma /= np.max(gamma)

    return gamma

Writing GER_CORE/ger_graph.py


In [3]:
import os

# 1. Cria a pasta física GER_CORE no painel esquerdo do Colab (se ela não existir)
pasta = "GER_CORE"
if not os.path.exists(pasta):
    os.makedirs(pasta)
    print(f"✅ Pasta '{pasta}' criada com sucesso no ambiente!")
else:
    print(f"ℹ️ A pasta '{pasta}' já existe no ambiente.")

# 2. Mostra onde ela está localizada
print("Diretório atual de trabalho:", os.getcwd())

✅ Pasta 'GER_CORE' criada com sucesso no ambiente!
Diretório atual de trabalho: /content


In [4]:
%%writefile GER_CORE/ger_potential.py

"""
=========================================================
GER CORE
Arquivo : ger_potential.py
=========================================================

Potenciais não lineares utilizados pela Geometria
Espectral Relacional.

Este módulo concentra toda a física não linear do
projeto.

Cada potencial retorna:

    força
    energia potencial

A interface pública é a classe:

    Potential.evaluate(...)
"""

from __future__ import annotations

import numpy as np


# =========================================================
# Potencial A
# =========================================================

def potential_A(gamma):
    """
    Potencial cúbico (φ⁴).

    V = γ⁴ / 4
    """

    force = gamma**3

    energy = 0.25 * gamma**4

    return force, energy


# =========================================================
# Potencial C
# =========================================================

def potential_C(gamma):
    """
    Potencial saturante.

    V = log(cosh(γ))
    """

    force = np.tanh(gamma)

    energy = np.log(np.cosh(gamma))

    return force, energy


# =========================================================
# Interface pública
# =========================================================

class Potential:
    """
    Interface única para todos os potenciais.

    Exemplo
    -------

    force, energy = Potential.evaluate(
        gamma,
        "A"
    )
    """

    @staticmethod
    def evaluate(gamma, potential="A"):

        potential = potential.upper()

        if potential == "A":

            return potential_A(gamma)

        elif potential == "C":

            return potential_C(gamma)

        raise ValueError(
            f"Potencial '{potential}' não reconhecido."
        )

Writing GER_CORE/ger_potential.py


In [5]:

%%writefile GER_CORE/ger_metrics.py

"""
=========================================================
GER CORE
Arquivo : ger_metrics.py
=========================================================

Métricas globais da Geometria Espectral Relacional.

Este módulo implementa:

• Energia Hamiltoniana
• Norma L²
• Amplitude máxima
• Erro relativo de energia
• Critério automático de divergência

Todas as auditorias utilizam estas rotinas.
"""

from __future__ import annotations

import numpy as np

from ger_potential import Potential


# =========================================================
# Configurações globais
# =========================================================

ENERGY_TOL = 1e-4

DIVERGENCE_AMPLITUDE = 1e6

EPS = 1e-15


# =========================================================
# Norma L²
# =========================================================

def compute_l2_norm(gamma):
    """
    Norma L² do campo.
    """

    gamma = np.asarray(gamma)

    return np.sqrt(np.sum(gamma**2))


# =========================================================
# Amplitude máxima
# =========================================================

def compute_max_amplitude(gamma):
    """
    Valor absoluto máximo do campo.
    """

    gamma = np.asarray(gamma)

    return np.max(np.abs(gamma))


# =========================================================
# Energia Hamiltoniana
# =========================================================

def compute_hamiltonian(
    gamma,
    velocity,
    laplacian,
    beta,
    potential="A"
):
    """
    Energia Hamiltoniana completa.

    H =
        Energia cinética
      + Energia elástica
      + Energia potencial não linear
    """

    gamma = np.asarray(gamma)

    velocity = np.asarray(velocity)

    kinetic = 0.5 * np.sum(velocity**2)

    elastic = 0.5 * gamma @ (laplacian @ gamma)

    _, potential_energy = Potential.evaluate(
        gamma,
        potential
    )

    nonlinear = beta * np.sum(potential_energy)

    return kinetic + elastic + nonlinear


# =========================================================
# Erro relativo de energia
# =========================================================

def relative_energy_error(
    reference_energy,
    current_energy
):
    """
    Erro relativo da energia Hamiltoniana.
    """

    return abs(
        current_energy - reference_energy
    ) / (
        abs(reference_energy) + EPS
    )


# =========================================================
# Critério automático de divergência
# =========================================================

def check_divergence(
    gamma,
    energy_error
):
    """
    Detecta automaticamente explosões numéricas.
    """

    amplitude = compute_max_amplitude(gamma)

    if amplitude > DIVERGENCE_AMPLITUDE:

        return True

    if np.isnan(amplitude):

        return True

    if np.isinf(amplitude):

        return True

    if np.isnan(energy_error):

        return True

    if np.isinf(energy_error):

        return True

    return False

Writing GER_CORE/ger_metrics.py


In [6]:
%%writefile GER_CORE/ger_modal.py

"""
=========================================================
GER CORE
Arquivo : ger_modal.py
=========================================================

Observatório Espectral da Geometria Espectral Relacional.

Este módulo contém todas as ferramentas de análise
modal utilizadas pelas auditorias S26-B.

Responsabilidades:

• Projeção na base espectral
• Distribuição de energia modal
• Entropia espectral
• Centro e largura modal
• Participação modal
• Separação por bandas espectrais
"""

from __future__ import annotations

import numpy as np


EPS = 1e-15


# =========================================================
# Projeção modal
# =========================================================

def modal_projection(
    gamma,
    eigenvectors
):
    """
    Projeta o campo na base espectral.

    gamma_hat = V^T gamma
    """

    gamma = np.asarray(gamma)

    return eigenvectors.T @ gamma


# =========================================================
# Energia modal
# =========================================================

def modal_energy(
    gamma,
    eigenvectors
):
    """
    Energia associada a cada modo espectral.
    """

    coefficients = modal_projection(
        gamma,
        eigenvectors
    )

    energy = coefficients**2

    return energy


# =========================================================
# Probabilidade modal
# =========================================================

def modal_probability(
    gamma,
    eigenvectors
):
    """
    Normalização da energia modal.

    Soma das probabilidades = 1
    """

    energy = modal_energy(
        gamma,
        eigenvectors
    )

    total = np.sum(energy) + EPS

    return energy / total


# =========================================================
# Modo dominante
# =========================================================

def dominant_mode(probability):
    """
    Retorna o modo com maior concentração.
    """

    return int(
        np.argmax(probability)
    )


# =========================================================
# Centro espectral
# =========================================================

def spectral_center(probability):
    """
    Centro médio dos modos.
    """

    modes = np.arange(
        len(probability)
    )

    return np.sum(
        modes * probability
    )


# =========================================================
# Largura espectral
# =========================================================

def spectral_width(probability):
    """
    Desvio modal da distribuição espectral.
    """

    modes = np.arange(
        len(probability)
    )

    center = spectral_center(
        probability
    )

    variance = np.sum(
        probability *
        (modes - center)**2
    )

    return np.sqrt(
        variance
    )


# =========================================================
# Entropia espectral
# =========================================================

def spectral_entropy(probability):
    """
    Entropia de Shannon da distribuição modal.
    """

    p = probability[
        probability > 0
    ]

    return -np.sum(
        p * np.log(p)
    )


# =========================================================
# Participation Ratio
# =========================================================

def participation_ratio(probability):
    """
    Mede quantos modos participam efetivamente.

    PR = 1 / Σp²
    """

    return 1.0 / (
        np.sum(probability**2)
        + EPS
    )


# =========================================================
# Energia por bandas
# =========================================================

def spectral_bands(
    probability
):
    """
    Divide energia espectral em:

    baixa frequência
    média frequência
    alta frequência
    """

    n = len(probability)

    low_end = n // 3
    mid_end = 2 * n // 3

    low = np.sum(
        probability[:low_end]
    )

    medium = np.sum(
        probability[low_end:mid_end]
    )

    high = np.sum(
        probability[mid_end:]
    )

    return {
        "low": low,
        "medium": medium,
        "high": high
    }


# =========================================================
# Auditoria completa
# =========================================================

def analyze_modal_state(
    gamma,
    eigenvectors
):
    """
    Executa todo o observatório espectral.

    Retorna todas as métricas modais.
    """

    probability = modal_probability(
        gamma,
        eigenvectors
    )

    return {

        "modal_energy":
            modal_energy(
                gamma,
                eigenvectors
            ),

        "probability":
            probability,

        "dominant_mode":
            dominant_mode(
                probability
            ),

        "modal_center":
            spectral_center(
                probability
            ),

        "modal_width":
            spectral_width(
                probability
            ),

        "spectral_entropy":
            spectral_entropy(
                probability
            ),

        "participation_ratio":
            participation_ratio(
                probability
            ),

        "spectral_bands":
            spectral_bands(
                probability
            )
    }

Writing GER_CORE/ger_modal.py


In [7]:
%%writefile GER_CORE/ger_snapshot.py

"""
=========================================================
GER CORE
Arquivo : ger_snapshot.py
=========================================================

Registro padronizado dos estados temporais da GER.

Cada snapshot representa uma fotografia completa
do sistema em um instante da evolução.

Responsabilidades:

• Métricas globais
• Observatório espectral
• Organização dos dados
"""

from __future__ import annotations


from ger_metrics import (
    compute_hamiltonian,
    compute_l2_norm,
    compute_max_amplitude
)

from ger_modal import (
    analyze_modal_state
)


# =========================================================
# Construção do Snapshot
# =========================================================

def build_snapshot(
    step,
    time,
    gamma,
    velocity,
    laplacian,
    beta,
    potential,
    eigenvectors,
    theta=None
):
    """
    Cria um registro completo do estado.

    Parameters
    ----------

    step :
        passo temporal

    time :
        tempo físico/numerico

    gamma :
        campo atual

    velocity :
        velocidade atual

    laplacian :
        operador discreto

    beta :
        intensidade não linear

    potential :
        tipo de potencial

    eigenvectors :
        base espectral
    """


    # -----------------------------
    # Métricas globais
    # -----------------------------

    energy = compute_hamiltonian(
        gamma,
        velocity,
        laplacian,
        beta,
        potential
    )

    l2 = compute_l2_norm(
        gamma
    )

    amplitude = compute_max_amplitude(
        gamma
    )


    # -----------------------------
    # Análise espectral
    # -----------------------------

    modal = analyze_modal_state(
        gamma,
        eigenvectors
    )


    # -----------------------------
    # Registro final
    # -----------------------------

    snapshot = {

        "step":
            step,

        "time":
            time,

        "energy":
            energy,

        "l2":
            l2,

        "amplitude":
            amplitude,

        "dominant_mode":
            modal["dominant_mode"],

        "modal_center":
            modal["modal_center"],

        "modal_width":
            modal["modal_width"],

        "spectral_entropy":
            modal["spectral_entropy"],

        "participation_ratio":
            modal["participation_ratio"],

        "probability":
            modal["probability"],

        "modal_energy":
            modal["modal_energy"],

        "spectral_bands":
            modal["spectral_bands"],

        "gamma":
            gamma.copy()
    }


    return snapshot

Writing GER_CORE/ger_snapshot.py
